In [ ]:
import pandas as pd
import numpy as np
import json

df1 = pd.read_csv("/kaggle/input/taco-cleaned/taco_cleaned.csv")
df2 = pd.read_csv("/kaggle/input/cleaned-datasets-of-alpaca/cleaned_data_2.csv")
df3 = pd.read_csv("/kaggle/input/leetcode-with-desciption-and-code/leetcode_with_description_and_code.csv")
df4 = pd.read_csv("/kaggle/input/python-dataset-for-dsa-questions/cleaned_flytec.csv")

In [ ]:
path_to_taco = './taco_cleaned.csv' #provide path to taco_cleaned.csv created using TACO_cleaning.ipynb
path_to_alpaca = './alpaca.csv' #provide path to alpaca.csv created using leetcode_data_preparation.ipynb
path_to_leetcode = './leetcode.csv' #provide path to flytech.csv created using flytech_cleaning.ipynb
path_to_flytech = './flytech.csv' #provide path to alpaca.csv created using alpaca_preparation.ipynb

In [ ]:
taco_df = pd.read_csv(path_to_taco)
alpaca_df = pd.read_csv(path_to_alpaca)
leetcode_df = pd.read_csv(path_to_leetcode)
flytech_df = pd.read_csv(path_to_flytech)

In [ ]:
alpaca_df = alpaca_df.iloc[200:1201]

In [ ]:
alpaca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 200 to 1200
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1001 non-null   int64  
 1   id                 1001 non-null   int64  
 2   title              1001 non-null   object 
 3   description        1001 non-null   object 
 4   is_premium         1001 non-null   int64  
 5   difficulty         1001 non-null   object 
 6   solution_link      546 non-null    object 
 7   acceptance_rate    1001 non-null   float64
 8   frequency          1001 non-null   float64
 9   url                1001 non-null   object 
 10  discuss_count      1001 non-null   int64  
 11  accepted           1001 non-null   object 
 12  submissions        1001 non-null   object 
 13  companies          964 non-null    object 
 14  related_topics     941 non-null    object 
 15  likes              1001 non-null   int64  
 16  dislikes           100

In [ ]:
alpaca_df = alpaca_df[['description', 'code']]

In [ ]:
flytech_df = flytech_df.iloc[:4000]

In [36]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  4000 non-null   object
 1   code         4000 non-null   object
 2   category     4000 non-null   object
dtypes: object(3)
memory usage: 93.9+ KB


In [37]:
df4 = df4[['instruction','code']]

In [ ]:
alpaca_df = alpaca_df.iloc[:5000]


In [ ]:
alpaca_df = alpaca_df[['instruction','output']]

In [ ]:
taco_df = taco_df.iloc[:15000]

In [ ]:
taco_df = taco_df[['question','solutions']]

In [ ]:
taco_df = taco_df.rename(columns={'question': 'instruction', 'solutions': 'code'})
alpaca_df = alpaca_df.rename(columns={'instruction': 'instruction', 'output': 'code'})
df3 = df3.rename(columns={'description': 'instruction', 'code': 'code'})

In [ ]:
combined_df = pd.concat([taco_df, alpaca_df, df3, df4], ignore_index=True)

In [46]:
combined_df.head()

,instruction,code
0,The city park of IT City contains n east to we...,n = int(input())\ncn5 = n * (n - 1) // 2 * (n ...
1,Zookeeper is buying a carton of fruit to feed ...,"hist = [0] * 1000005\n\ndef solve(n, s):\n\tto..."
2,Sasha and Kolya decided to get drunk with Coke...,from collections import deque\nMAX_A = 1000\n\...
3,"Read problem statements in [Hindi], [Bengali],...","p = 10 ** 9 + 7\n\ndef power(a, n):\n\tres = 1..."
4,"I started this as a joke among friends, tellin...",from math import *\nDIGS = '0123456789ABCDEFGH...


In [47]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25001 entries, 0 to 25000
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  25001 non-null  object
 1   code         25001 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB


In [48]:
shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [49]:
train = shuffled_df.iloc[0:18000]

In [50]:
validation = shuffled_df.iloc[18000:21500]

In [51]:
test = shuffled_df.iloc[21500:]

In [52]:
test.head()

,instruction,code
21500,William has a favorite bracket sequence. Since...,"from os import path\nfrom io import BytesIO, I..."
21501,"You are given an array [A_{1}, \ldots, A_{N}]....",t = int(input())\nwhile t > 0:\n\tt -= 1\n\tn ...
21502,Write a program to calculate values of arithme...,"OVER = 10000\n\ndef factor():\n\tglobal w, pos..."
21503,Manasa is a student in the department of Mathe...,P = 10 ** 9 + 7\nn = int(input())\ndata = [map...
21504,"Read problems statements in Mandarin chinese, ...",def f(h):\n\tif h == 1:\n\t\treturn 1\n\telif ...


In [56]:
import json

SYSTEM_PROMPT = """You are an expert Python programmer. Your task is to generate correct, efficient Python code.
- Write only the function implementation
- Do not include markdown code blocks
- Do not include explanations
- The code must be syntactically correct
- There must be proper indentation.
- you will be given function signature
- create the program that fits into the function signature.
- always enclose code only between ``` and ```
- Do not reason
- Do not give any examples
"""

def format_llama(inst, code):
    return (
        "<s>[INST] <<SYS>>\n"
        f"{SYSTEM_PROMPT}\n"
        "<</SYS>>\n"
        f"{inst} [/INST] ```\n{code}``` </s>"
    )

# Write to JSONL
with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, row in train.iterrows():
        formatted_text = format_llama(
            str(row["instruction"]),
            str(row["code"])
        )
        obj = {"text": formatted_text}
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("JSONL file created: validation.jsonl")

JSONL file created: validation.jsonl


In [57]:
import json

SYSTEM_PROMPT = """You are an expert Python programmer. Your task is to generate correct, efficient Python code.
- Write only the function implementation
- Do not include markdown code blocks
- Do not include explanations
- The code must be syntactically correct
- There must be proper indentation.
- you will be given function signature
- create the program that fits into the function signature.
- always enclose code only between ``` and ```
- Do not reason
- Do not give any examples
"""

def format_llama(inst, code):
    return (
        "<s>[INST] <<SYS>>\n"
        f"{SYSTEM_PROMPT}\n"
        "<</SYS>>\n"
        f"{inst} [/INST] ```\n{code}``` </s>"
    )

# Write to JSONL
with open("validation.jsonl", "w", encoding="utf-8") as f:
    for _, row in validation.iterrows():
        formatted_text = format_llama(
            str(row["instruction"]),
            str(row["code"])
        )
        obj = {"text": formatted_text}
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("JSONL file created: validation.jsonl")

JSONL file created: validation.jsonl


In [58]:
import json

SYSTEM_PROMPT = """You are an expert Python programmer. Your task is to generate correct, efficient Python code.
- Write only the function implementation
- Do not include markdown code blocks
- Do not include explanations
- The code must be syntactically correct
- There must be proper indentation.
- always enclose code only between ``` and ```
- Do not reason
- Do not give any examples
"""

def format_llama(inst, code):
    return (
        "<s>[INST] <<SYS>>\n"
        f"{SYSTEM_PROMPT}\n"
        "<</SYS>>\n"
        f"{inst} [/INST] ```\n{code}``` </s>"
    )

# Write to JSONL
with open("test.jsonl", "w", encoding="utf-8") as f:
    for _, row in test.iterrows():
        formatted_text = format_llama(
            str(row["instruction"]),
            str(row["code"])
        )
        obj = {"text": formatted_text}
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("JSONL file created: validation.jsonl")

JSONL file created: validation.jsonl
